In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import time
import joblib
import pandas as pd

sys.path.append('../src')

## Сэмплы данных

Типичные данные по формату тестового датасета:

In [3]:
data_ft = pd.read_parquet('./data/data_ft.parquet')

In [4]:
data_ft.head()

,session_id,final_metric_asessor_1,final_metric_asessor_2,final_metric_asessor_3,final_metric_asessor_4,final_metric_asessor_5,estimation,query_id,input_query,output_answer
0,0,1.0,NaN,NaN,NaN,1.0,1,b50744e833cefe610331f3d26210ace6,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...
1,1,NaN,NaN,NaN,0.0,1.0,0,6bca9eec0de89a8dab6b1f77bb1116a5,анализ моего портфеля за 25 год,"К сожалению, из-за технических проблем не удал..."
2,2,1.0,NaN,0.0,NaN,NaN,0,d1de6397ed80dec217cd79a0ca5fb742,инвесткопилка,В вашем портфеле отсутствуют данные о продукте...
3,3,1.0,1.0,NaN,NaN,NaN,1,5908a95011350ff5ce48585d528d6d54,Фонд Рублевые сбережения,"В вашем портфеле отсутствуют данные о фонде ""Р..."
4,4,NaN,1.0,NaN,NaN,1.0,1,486cdbf61e53fb5a97726d7bcaa4b359,Структура портфеля и риски,Основные характеристики портфеля:\n- Общий объ...


Данные, в которых все - и трейсы, и разметка - слито вместе:

In [5]:
data_traces = pd.read_parquet('data/data_traces.parquet')

In [6]:
data_traces.head()

,session_id,final_metric_assessor_1,final_metric_assessor_2,final_metric_assessor_3,final_metric_assessor_4,final_metric_assessor_5,estimation,input_query,output_answer,traceid,spanid,name,starttimeunixnano,endtimeunixnano
0,0.0,1.0,NaN,NaN,NaN,1.0,1.0,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...,b50744e833cefe610331f3d26210ace6,ec6edde186b2ad4a,nodes_classifier,1.765888e+18,1.765888e+18
1,0.0,1.0,NaN,NaN,NaN,1.0,1.0,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...,b50744e833cefe610331f3d26210ace6,dc564dc99cf69022,<unknown>,1.765888e+18,1.765888e+18
2,0.0,1.0,NaN,NaN,NaN,1.0,1.0,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...,b50744e833cefe610331f3d26210ace6,c8afd12e139fd613,fewshots_node,1.765888e+18,1.765888e+18
3,0.0,1.0,NaN,NaN,NaN,1.0,1.0,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...,b50744e833cefe610331f3d26210ace6,350334e70db9f9b8,PromptTemplate,1.765888e+18,1.765888e+18
4,0.0,1.0,NaN,NaN,NaN,1.0,1.0,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...,b50744e833cefe610331f3d26210ace6,ec6edde186b2ad4a,nodes_classifier,1.765888e+18,1.765888e+18


Данные, где трейсы и разметка получаются отдельно в разных файлах:

In [7]:
data_traces2 = pd.read_parquet('./data/data_traces_short.parquet')

In [8]:
data_traces2.head()

,trace_id,session_id,llm_model,llm_total_token,starttime,endtime
0,5779740f8c1f81693cce291a301bb9c2,e2bde25f8b4842b8,GigaChat:1.0.26.20,1235,2025-12-17 04:04:48,2025-12-17 04:04:48
1,049cb99a5a1c6d1051809e3e8d8fdb4b,7ffb9f78d9da7e16,GigaChat:1.0.26.20,2508,2025-12-16 17:56:39,2025-12-16 17:56:46
2,ee61f8e797300f15f3607c4115c8c9b8,c22d7be48b105304,GigaChat:1.0.26.20,2696,2025-12-16 18:43:51,2025-12-16 18:43:56
3,6b88ebe37ed02632e977fc5cfcb3f50b,63f41ea85eefcc13,GigaChat:1.0.26.20,1239,2025-12-16 16:02:30,2025-12-16 16:02:30
4,70d817c06d419ed5199dae26c1633f45,5d5546ea95f8d27e,GigaChat:1.0.26.20,2859,2025-12-17 00:34:37,2025-12-17 00:34:37


In [9]:
data_ft_full = pd.read_parquet('./data/data_ft_full.parquet')

In [10]:
data_ft_full.head()

,session_id,final_metric_asessor_1,final_metric_asessor_2,final_metric_asessor_3,final_metric_asessor_4,final_metric_asessor_5,estimation,query_id,input_query,output_answer
0,0,1.0,NaN,NaN,NaN,1.0,1,b50744e833cefe610331f3d26210ace6,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...
1,1,NaN,NaN,NaN,0.0,1.0,0,6bca9eec0de89a8dab6b1f77bb1116a5,анализ моего портфеля за 25 год,"К сожалению, из-за технических проблем не удал..."
2,2,1.0,NaN,0.0,NaN,NaN,0,d1de6397ed80dec217cd79a0ca5fb742,инвесткопилка,В вашем портфеле отсутствуют данные о продукте...
3,3,1.0,1.0,NaN,NaN,NaN,1,5908a95011350ff5ce48585d528d6d54,Фонд Рублевые сбережения,"В вашем портфеле отсутствуют данные о фонде ""Р..."
4,4,NaN,1.0,NaN,NaN,1.0,1,486cdbf61e53fb5a97726d7bcaa4b359,Структура портфеля и риски,Основные характеристики портфеля:\n- Общий объ...


Данные с работой RAG:

In [11]:
data_rag = pd.read_pickle('data/data_rag.pkl')

In [12]:
data_rag.head()

,question,prompt,true_answer,model_answer,accuracy,completeness,context,sim_cosine_distance,thema,passage_accuracy
0,Сколько времени уходит на изготовление куклы?,Ты - эксперт по романам Рэя Брэдбери.\n ...,На изготовление куклы уходит 3 часа,3 часа,{1: 1},{1: 0},[Прошел целый час. Джорджа Хилла уже давно про...,"[0.14784731518704508, 0.1963453930506014, 0.17...",Рэй Брэдберри- Наказание без преступления,"{1: [1, 0, 0], 2: [1, 1, 0], 3: [1, 1, 1]}"
1,Какого цвета волосы у Кэтрин?,Ты - эксперт по романам Рэя Брэдбери.\n ...,У Кэтрин волосы золотого цвета,Золотого,{1: 1},{1: 0},"[кушетку, и давление его тела заставило вращат...","[0.1959770419693152, 0.18542409594256082, 0.19...",Рэй Брэдберри- Наказание без преступления,"{1: [1, 1, 0], 2: [1, 1, 0], 3: [1, 1, 1]}"
2,Почему Джордж проснулся в слезах?,Ты - эксперт по романам Рэя Брэдбери.\n ...,"Джордж проснулся в слезах после кошмара, в кот...","Джорджу приснился кошмар, в котором Кэтрин был...",{1: 1},{1: 1},"[молода! Почему, почему? .. \nЭта сцена навсег...","[0.1653891186394244, 0.16783296628757804, 0.14...",Рэй Брэдберри- Наказание без преступления,"{1: [1, 1, 1], 2: [1, 1, 0], 3: [0, 1, 0]}"
3,С чем Джордж сравнил ногти Кэтрин?,Ты - эксперт по романам Рэя Брэдбери.\n ...,Джордж сравнивал ногти Кэтрин с морскими раков...,С морскими раковинами,{1: 1},{1: 0},[голубиные под кудрями твоими... Как лента ала...,"[0.171377728813355, 0.17755472840018538, 0.174...",Рэй Брэдберри- Наказание без преступления,"{1: [1, 0, 0], 2: [1, 0, 0], 3: [0, 0, 0]}"
4,Получилось ли у Джона убить Кэтрин?,Ты - эксперт по романам Рэя Брэдбери.\n ...,"Нет, Джордж убил куклу в виде Кэтрин","Да, Джордж убил Кэтрин",{1: 0},{1: 0},[- Вы хотите убить свою жену? - спросил темнов...,"[0.14638012157375146, 0.14933457369494585, 0.1...",Рэй Брэдберри- Наказание без преступления,"{1: [1, 0, 0], 2: [1, 1, 1], 3: [1, 1, 1]}"


## Принципы работы с sampler'ом

Главной задачей sampler'а является хранение и предобработка данных. Предобработка реализована в конструкторе. Реализация предобработки зависит от конкретного датасета и того формата данных, которые прислали контрагенты. 

Тестовый датасет должен лежать в аттрибуте `test` (он есть всегда), валидационный - в аттрибуте `val`, а обучающий - в `train`. 

Датасет состоит из трех частей: 

`X` - dataFrame с вопросами, ответами, контекстом и т.п.; 
`y` - dataFrame, необходимый для подсчета метрик; 
`traces` - dataFrame с распаршенными трейсами формата aef-tracing (опциональный).

Существует 2 способа работы с семплером:
1. Написать предобработку данных вне сэмплера и запихнуть данные в какой-то уже имеющийся в библиотеке сэмплер (***Нецелевой***);
2. Создать класс, в котором описать логику обработки данных (***Целевой***).

Для каждого формата датасетов определена строгая структура для корректной работы с библиотекой `llm_val`.

**Отдельные запросы**:
- ***X***
    1. `session_id` - Идентификатор сессии взаимодействия пользователя с агентом. `Union[int, uuid]`;
    2. `query_id` - Идентификатор отдельного запроса пользователя. `Union[int, str, uuid]`;
    3. `input_query` - Текст запроса в GenAI-решение. `str`;
    4. `output_answer` - Текст ответа GenAI-решения на входной запрос. `str`;
    5. `reference_answer` - Истинный ответ (ground truth) или ответ, на который стоит ориентироваться при оценке качества ответа от GenAI-решения. `str` (N.B. Если таких возможно несколько на один запрос, то нужно использовать нейминг `reference_answer_1`, `reference_answer_2`, ...);
    6. `[metric_name]_metric` - Значение метрики качества или критерия оценки `metric_name`, проставляемой одним асессором для ответа GenAI-решения. `float`;
    7. `[module_name]_input_query` - Текст запроса в конкретный модуль `module_name` GenAI-решение. `str`;
    8. `[module_name]_output_answer` - Текст ответа от конкретного модуля `module_name` GenAI-решения на входной запрос. `str`;
    9. `[module_name]_reference_answer` - Истинный ответ (ground truth) или ответ, на который стоит ориентироваться при оценке качества ответа от конкретного модуля `module_name` GenAI-решения. `str` (N.B. Нейминг для нескольких ответов аналогичен `reference_answer` выше);
    10. `[module_name]_[metric_name]_metric` - Значение метрики качества или критерия оценки, проставляемой одним асессором для ответа от конкретного модуля `module_name` GenAI-решения. `float`;
    11. `assessor_id` - идентификатор асессора. `Union[int, uuid]`;
    12. `scenario` - Тематика запроса / уровень сложности запроса / название тестового сценария / другие классифицирующие запрос характеристики. `str`.

- ***y***

    
    **Индекс**: `session_id` - Идентификатор сессии взаимодействия пользователя с агентом. `Union[int, uuid]`;
    1. `[metric_name]_metric` - Посчитанная метрика качества `metric_name` на конкретном вопросе: либо агрегированный результат столбца `[metric_name]_metric` в ***X***, либо посчитанная автоматически. `Union[float, int]`.

**Диалог**:
- ***X***
    1. `session_id` - Идентификатор сессии взаимодействия пользователя с агентом. `Union[int, uuid]`;
    2. `dialogue` - Полный текст диалога между пользователей и GenAI-решением в рамках сессии. Представим как список из кортежей, где каждый кортеж имеет вид `(str, str, str)`. В каждом кортеже есть строки - `query_id`, `input_query`, `output_answer`. `List[(str, str, str)]`;
    3. `[metric_name]_metric` - Значение метрики качества или критерия оценки `metric_name`, проставляемой одним асессором для ответа GenAI-решения. `float`;
    4. `assessor_id` - идентификатор асессора. `Union[int, uuid]`; 
    5. `scenario` - Тематика запроса / уровень сложности запроса / название тестового сценария / другие классифицирующие запрос характеристики. `str`.
    
- ***y***

    
    **Индекс**: `session_id` - Идентификатор сессии взаимодействия пользователя с агентом. `Union[int, uuid]`;
    1. `[metric_name]_metric` - Посчитанная метрика качества `metric_name` на конкретном вопросе: либо агрегированный результат столбца `[metric_name]_metric` в ***X***, либо посчитанная автоматически. `Union[float, int]`.

**Трейсы** - опционально можно добавить в сэмплер для любого формата датасета.
- ***X***, ***y***: описано в других пунктах;
- ***traces***:
    1. `trace_id` - id трейса, соответствующее одному диалогу/запросу. `Union[int, uuid]`;
    2. `span_id` - id спана, соответствующее одной ноде/инструменту агента. `Union[int, uuid]`;
    3. `span_name` - название ноды/инструмента агента. `str`;
    4. `start_time_ns` - время начала работы ноды агента в unix-формате. `datetime64[ns]`;
    5. `end_time_ns` - время окончания работы ноды агента в unix-формат. `datetime64[ns]`;
    6. `session_id` - (идентично полю в `X`) - идентификатор сессии взаимодействия пользователя с агентом. `Union[int, uuid]`.
    
**RAG**:
- ***X***:
    1. `session_id` - Идентификатор сессии взаимодействия пользователя с агентом. `Union[int, uuid]`;
    2. `query_id` - Идентификатор отдельного запроса пользователя. `Union[int, str, uuid]`;
    3. `input_query` - Текст запроса в GenAI-решение. `str`;
    4. `output_answer` - Текст ответа GenAI-решения на входной запрос. `str`;
    5. `reference_answer` - Истинный ответ (ground truth) или ответ, на который стоит ориентироваться при оценке качества ответа от GenAI-решения. `str` (N.B. Если таких возможно несколько на один запрос, то нужно использовать нейминг `reference_answer_1`, `reference_answer_2`, ...);
    6. `[metric_name]_metric` - Значение метрики качества или критерия оценки `metric_name`, проставляемой одним асессором для ответа GenAI-решения. `float`;
    7. `passage_accuracy` - Словарь с оценками релевантностей пассажей. `Dict[Union[str, int], List[int]]`;
    8. `scenario` - Тематика запроса / уровень сложности запроса / название тестового сценария / другие классифицирующие запрос характеристики. `str`.

- ***y***

    
    **Индекс**: `session_id` - Идентификатор сессии взаимодействия пользователя с агентом. `Union[int, uuid]`;
    1. `sim_<название_расстояния>` - Расстояния от вопроса до пассажей. Размер списка n, где n - количество пассажей. `List[float]`;
    2. `[metric_name]_metric` - Посчитанная метрика качества `metric_name` на конкретном вопросе: либо агрегированный результат столбца `[metric_name]_metric` в ***X***, либо посчитанная автоматически. `Union[float, int]`.

## Sampler

Простейшим sampler'ом является `Sampler`. В нем нет никакой предобработки, он только хранит данные:

In [13]:
from llm_val.sampler.Sampler import Sampler

In [14]:
sampler = Sampler(test={'X': data_ft, 'y': None})

In [15]:
sampler.test['X'].head()

,session_id,final_metric_asessor_1,final_metric_asessor_2,final_metric_asessor_3,final_metric_asessor_4,final_metric_asessor_5,estimation,query_id,input_query,output_answer
0,0,1.0,NaN,NaN,NaN,1.0,1,b50744e833cefe610331f3d26210ace6,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...
1,1,NaN,NaN,NaN,0.0,1.0,0,6bca9eec0de89a8dab6b1f77bb1116a5,анализ моего портфеля за 25 год,"К сожалению, из-за технических проблем не удал..."
2,2,1.0,NaN,0.0,NaN,NaN,0,d1de6397ed80dec217cd79a0ca5fb742,инвесткопилка,В вашем портфеле отсутствуют данные о продукте...
3,3,1.0,1.0,NaN,NaN,NaN,1,5908a95011350ff5ce48585d528d6d54,Фонд Рублевые сбережения,"В вашем портфеле отсутствуют данные о фонде ""Р..."
4,4,NaN,1.0,NaN,NaN,1.0,1,486cdbf61e53fb5a97726d7bcaa4b359,Структура портфеля и риски,Основные характеристики портфеля:\n- Общий объ...


## FastTrackSampler

Тут сэмплер создается по данным с разметкой без трейсов:

In [16]:
from llm_val.sampler.FastTrackSampler import FastTrackSampler

In [17]:
sampler_ft = FastTrackSampler(data_ft)

In [18]:
sampler_ft.test['X'].head()

,final_metric,assessor_id,session_id,query_id,input_query,output_answer
0,1.0,1,0,b50744e833cefe610331f3d26210ace6,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...
1,NaN,1,1,6bca9eec0de89a8dab6b1f77bb1116a5,анализ моего портфеля за 25 год,"К сожалению, из-за технических проблем не удал..."
2,1.0,1,2,d1de6397ed80dec217cd79a0ca5fb742,инвесткопилка,В вашем портфеле отсутствуют данные о продукте...
3,1.0,1,3,5908a95011350ff5ce48585d528d6d54,Фонд Рублевые сбережения,"В вашем портфеле отсутствуют данные о фонде ""Р..."
4,NaN,1,4,486cdbf61e53fb5a97726d7bcaa4b359,Структура портфеля и риски,Основные характеристики портфеля:\n- Общий объ...


In [19]:
sampler_ft.test['y'].head()

,final_metric
session_id,
0,1
2,1
3,1
7,0
8,0


## TracesSampler

Пример сэмплера, который отрабатывает на данных, содержащих в себе и разметку, и трейсы:

In [20]:
from llm_val.sampler.TracesSampler import TracesSampler

In [21]:
traces_sampler = TracesSampler(data_traces)

In [22]:
traces_sampler.test['X'].head()

,final_metric_assessor_4,final_metric_assessor_1,trace_id,final_metric_assessor_5,span_id,input_query,session_id,final_metric_assessor_2,output_answer,final_metric_assessor_3
0,NaN,1.0,b50744e833cefe610331f3d26210ace6,1.0,ec6edde186b2ad4a,Расскажи о состоянии портфеля за месяц,0.0,NaN,Стоимость портфеля за ноябрь-декабрь 2025 года...,NaN
1,NaN,1.0,b50744e833cefe610331f3d26210ace6,1.0,dc564dc99cf69022,Расскажи о состоянии портфеля за месяц,0.0,NaN,Стоимость портфеля за ноябрь-декабрь 2025 года...,NaN
2,NaN,1.0,b50744e833cefe610331f3d26210ace6,1.0,c8afd12e139fd613,Расскажи о состоянии портфеля за месяц,0.0,NaN,Стоимость портфеля за ноябрь-декабрь 2025 года...,NaN
3,NaN,1.0,b50744e833cefe610331f3d26210ace6,1.0,350334e70db9f9b8,Расскажи о состоянии портфеля за месяц,0.0,NaN,Стоимость портфеля за ноябрь-декабрь 2025 года...,NaN
4,NaN,1.0,b50744e833cefe610331f3d26210ace6,1.0,381642cee078975d,Расскажи о состоянии портфеля за месяц,0.0,NaN,Стоимость портфеля за ноябрь-декабрь 2025 года...,NaN


In [23]:
traces_sampler.test['y'].head()

,estimation
session_id,
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0


In [24]:
traces_sampler.test['traces'].head()

,session_id,trace_id,span_id,span_name,start_time_ns,end_time_ns
0,0.0,b50744e833cefe610331f3d26210ace6,ec6edde186b2ad4a,nodes_classifier,2025-12-16 12:22:03.396299264,2025-12-16 12:22:03.398770944
1,0.0,b50744e833cefe610331f3d26210ace6,dc564dc99cf69022,<unknown>,2025-12-16 12:22:03.526691584,2025-12-16 12:22:03.528349696
2,0.0,b50744e833cefe610331f3d26210ace6,c8afd12e139fd613,fewshots_node,2025-12-16 12:22:03.401039616,2025-12-16 12:22:03.529744896
3,0.0,b50744e833cefe610331f3d26210ace6,350334e70db9f9b8,PromptTemplate,2025-12-16 12:22:03.540996096,2025-12-16 12:22:03.542539776
56,0.0,b50744e833cefe610331f3d26210ace6,381642cee078975d,GigaChat,2025-12-16 12:22:03.544433152,2025-12-16 12:22:12.636401152


## TracesSampler2

Пример сэмплера, который собирается из двух отдельных файлов - разметки и трейсы:

In [25]:
from llm_val.sampler.TracesSampler2 import TracesSampler2

In [26]:
sampler_traces2 = TracesSampler2(data_ft_full, data_traces2)

In [27]:
sampler_traces2.test['X'].head()

,final_metric,assessor_id,session_id,query_id,input_query,output_answer
0,1.0,1,b50744e833cefe610331f3d26210ace6,b50744e833cefe610331f3d26210ace6,Расскажи о состоянии портфеля за месяц,Стоимость портфеля за ноябрь-декабрь 2025 года...
1,NaN,1,6bca9eec0de89a8dab6b1f77bb1116a5,6bca9eec0de89a8dab6b1f77bb1116a5,анализ моего портфеля за 25 год,"К сожалению, из-за технических проблем не удал..."
2,1.0,1,d1de6397ed80dec217cd79a0ca5fb742,d1de6397ed80dec217cd79a0ca5fb742,инвесткопилка,В вашем портфеле отсутствуют данные о продукте...
3,1.0,1,5908a95011350ff5ce48585d528d6d54,5908a95011350ff5ce48585d528d6d54,Фонд Рублевые сбережения,"В вашем портфеле отсутствуют данные о фонде ""Р..."
4,NaN,1,486cdbf61e53fb5a97726d7bcaa4b359,486cdbf61e53fb5a97726d7bcaa4b359,Структура портфеля и риски,Основные характеристики портфеля:\n- Общий объ...


In [28]:
sampler_traces2.test['y'].head()

,estimation
session_id,
b50744e833cefe610331f3d26210ace6,1
6bca9eec0de89a8dab6b1f77bb1116a5,0
d1de6397ed80dec217cd79a0ca5fb742,0
5908a95011350ff5ce48585d528d6d54,1
486cdbf61e53fb5a97726d7bcaa4b359,1


In [29]:
sampler_traces2.test['traces'].head()

,trace_id,session_id,llm_model,llm_total_token,start_time_ns,end_time_ns,llm_prompt_tokens
0,5779740f8c1f81693cce291a301bb9c2,5779740f8c1f81693cce291a301bb9c2,GigaChat:1.0.26.20,1235,2025-12-17 04:04:48,2025-12-17 04:04:48,1235
1,049cb99a5a1c6d1051809e3e8d8fdb4b,049cb99a5a1c6d1051809e3e8d8fdb4b,GigaChat:1.0.26.20,2508,2025-12-16 17:56:39,2025-12-16 17:56:46,2508
2,ee61f8e797300f15f3607c4115c8c9b8,ee61f8e797300f15f3607c4115c8c9b8,GigaChat:1.0.26.20,2696,2025-12-16 18:43:51,2025-12-16 18:43:56,2696
3,6b88ebe37ed02632e977fc5cfcb3f50b,6b88ebe37ed02632e977fc5cfcb3f50b,GigaChat:1.0.26.20,1239,2025-12-16 16:02:30,2025-12-16 16:02:30,1239
4,70d817c06d419ed5199dae26c1633f45,70d817c06d419ed5199dae26c1633f45,GigaChat:1.0.26.20,2859,2025-12-17 00:34:37,2025-12-17 00:34:37,2859


## RAGSamplerAgent

In [31]:
from llm_val.sampler.RAGSamplerAgent import RAGSamplerAgent

In [33]:
rag_sampler = RAGSamplerAgent(data_rag)

In [34]:
rag_sampler.test['X']

,system,question,context,scenario,true_answer,model_answer,accuracy,completeness,passage_accuracy
0,Ты - эксперт по романам Рэя Брэдбери.\n ...,Сколько времени уходит на изготовление куклы?,[Прошел целый час. Джорджа Хилла уже давно про...,Рэй Брэдберри- Наказание без преступления,На изготовление куклы уходит 3 часа,3 часа,{1: 1},{1: 0},"{1: [1, 0, 0], 2: [1, 1, 0], 3: [1, 1, 1]}"
1,Ты - эксперт по романам Рэя Брэдбери.\n ...,Какого цвета волосы у Кэтрин?,"[кушетку, и давление его тела заставило вращат...",Рэй Брэдберри- Наказание без преступления,У Кэтрин волосы золотого цвета,Золотого,{1: 1},{1: 0},"{1: [1, 1, 0], 2: [1, 1, 0], 3: [1, 1, 1]}"
2,Ты - эксперт по романам Рэя Брэдбери.\n ...,Почему Джордж проснулся в слезах?,"[молода! Почему, почему? .. \nЭта сцена навсег...",Рэй Брэдберри- Наказание без преступления,"Джордж проснулся в слезах после кошмара, в кот...","Джорджу приснился кошмар, в котором Кэтрин был...",{1: 1},{1: 1},"{1: [1, 1, 1], 2: [1, 1, 0], 3: [0, 1, 0]}"
3,Ты - эксперт по романам Рэя Брэдбери.\n ...,С чем Джордж сравнил ногти Кэтрин?,[голубиные под кудрями твоими... Как лента ала...,Рэй Брэдберри- Наказание без преступления,Джордж сравнивал ногти Кэтрин с морскими раков...,С морскими раковинами,{1: 1},{1: 0},"{1: [1, 0, 0], 2: [1, 0, 0], 3: [0, 0, 0]}"
4,Ты - эксперт по романам Рэя Брэдбери.\n ...,Получилось ли у Джона убить Кэтрин?,[- Вы хотите убить свою жену? - спросил темнов...,Рэй Брэдберри- Наказание без преступления,"Нет, Джордж убил куклу в виде Кэтрин","Да, Джордж убил Кэтрин",{1: 0},{1: 0},"{1: [1, 0, 0], 2: [1, 1, 1], 3: [1, 1, 1]}"


In [35]:
rag_sampler.test['y']

,passage_accuracy_metric,sim_cosine_distance,accuracy_metric,completeness_metric
0,"[1, 0, 0]","[0.14784731518704508, 0.1963453930506014, 0.17...",1.0,0.0
1,"[1, 1, 0]","[0.1959770419693152, 0.18542409594256082, 0.19...",1.0,0.0
2,"[0, 1, 0]","[0.1653891186394244, 0.16783296628757804, 0.14...",1.0,1.0
3,"[0, 0, 0]","[0.171377728813355, 0.17755472840018538, 0.174...",1.0,0.0
4,"[1, 0, 0]","[0.14638012157375146, 0.14933457369494585, 0.1...",0.0,0.0
